# **Lab 7 - Visión Por Computadora**

Integrantes:
- Francis Aguilar, 22243
- César López, 22535
- Jose Marchena, 22398

Repo:
https://github.com/MarchMol/lab7_vision

-----------------------
## **Instrucciones**
Usted forma parte del equipo de Inteligencia Artificial de MediScan Guatemala, una empresa de tecnología
médica que está desarrollando un sistema de apoyo al diagnóstico para médicos de primer nivel en clínicas
rurales. El sistema debe identificar posibles anomalías en radiografías de tórax (p. ej., presencia de
opacidades que puedan indicar neumonía). La restricción principal: las clínicas tienen conectividad a internet
intermitente, por lo que el modelo debe ser liviano y capaz de correr on-premise en una computadora
estándar sin GPU dedicada. Su equipo dispone de un dataset limitado de radiografías etiquetadas.
Para este laboratorio utilizará el siguiente dataset de dominio público.

Dataset: Chest X-Ray Images (Pneumonia)

https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia

-----------------
## **Task 1**
El siguiente conjunto de preguntas evalúa su capacidad de analizar, justificar y argumentar decisiones técnicas de entrenamiento en un contexto médico real. No basta con mencionar fórmulas; se espera que usted conecte cada concepto con sus implicaciones prácticas dentro del proyecto MediScan Guatemala.

### Pregunta 1.1
El médico coordinador del proyecto le presenta la siguiente situación: "Tenemos 800 radiografías etiquetadas por nuestros radiólogos. Un colega me dijo que con tan pocos datos el modelo va a memorizar todo y no va a servir para nada en producción."

Como ingeniero de IA a cargo, usted decide aplicar Data Augmentation como parte de la solución. Sin embargo, el médico le pregunta: "¿No estamos inventando datos falsos que podrían confundir al modelo?" Con esto en mente, responda las siguientes preguntas en su reporte:

1. Explíquele al médico, en términos que él pueda entender, qué es el Data Augmentation y por qué las imágenes generadas no son datos falsos. Use la analogía que considere más apropiada.

> **R//**
> Data augmentation significa cambiar atributos no intrinsecos de la data original para generar mas casos para el aprendizaje. Un ejemplo de esto podria ser el zoom. Si comenzamos con una foto original que indica un diagnostico positivo, el modelo deberia ser capaz de identificarla como tal incluso si la imagen tiene un zoom distinto, pues sigue siendo el mismo diagnostico. No estamos inventando data, estamos cambiano diferentes aspectos del contexto de nuestra informacion inicial para enriquecer el entrenamiento del modelo y evitar que haga conexiones muy simples.

> Una analogia seria como las tareas en una clase de matematicas. El profesor enseña formulas con ejemplos concretos durante clases y los estudiantes pueden aprender con estas. Data augmentation es como cuando el profesor deja una tarea. En la tarea estan las mismas formulas formula pero con diferentes escenarios, problemas, etc. Es decir, el contenido/logica/procedimiento de la formula no cambia, pero los diferentes ejemplos le pueden dar mas contexto a los estudiantes IDENTIFICAR cuando y cómo usar la formula; para que no se aprendan unicamente los ejemplos inciales que el profesor dio en clase.

2. En el contexto específico de radiografías de tórax, proponga tres transformaciones de Data Augmentation que serían válidas y justifique cada una. Luego, identifique una transformación que no debería aplicarse en este dominio médico y explique por qué podría comprometer la integridad diagnóstica del modelo.

> **R//**
> Transformaciones validas podrian ser Rotaciones (donde se le aplica un angulo al azar de rotacion dentro de un umbral), Shifting (mover la imagen ligeramente para simular posiciones diferentes de distintos pacientes) y Zoom/scaling (acercar o cortar una imagen para cambiarel foco y posicion de la informacion disponib;e)
> Segun se investigo, transformaciones que no se recomendarian serian cosas como flipping, pues conllevan a casos que son atomaicamente incorrectos. En el caso del torax, una transformacion flipping podria llevar a casos donde ciertos pacientes tienen el corazon del lado incorrecto. Otras que tampoco se recomiendan son las que comprometan la calidad de las imagenes, como blurs, recortes extremos, o cambios en contraste

3. ¿Es el Data Augmentation suficiente por sí solo para garantizar que el modelo generalice bien? Argumente su posición considerando otras variables del proceso de entrenamiento.
> **R//**
> Por si solo Data Augmentation no implica que siempre podra generalizar un modelo. Pero si es un paso en la escalera para desarrollar un modelo adecuado. Cuestiones de arquitectura como escoger correctamente el optimizador, la metrica de perdida, penalizaciones y otras tecnicas de regulacion como dropout, contribuyen an un modelo mas robusto. Sim ebargo siempre hay muchas cosas que tomar en cuenta como incluso la calidad de la informacion original, la validez de nuestra ground truth, la validacion que aplicamos y la arquitectura misma del modelo y sus capas.

### Pregunta 1.2
Durante el entrenamiento del modelo de MediScan Guatemala, el equipo registra las siguientes curvas de pérdida (loss) al finalizar 25 épocas:

| Época | Loss de Entrenamiento | Loss de Validación |
| --- | --- | --- |
5 | 0.52 | 0.55
10 | 0.31 | 0.38
15 | 0.18 | 0.42
20 | 0.09 | 0.61
25 | 0.04 | 0.89

Con base a estos datos, responda dentro de su reporte:

4. Identifique a partir de qué época aproximada comienza el sobreajuste (overfitting) y describa cómo se evidencia en los números de la tabla.
> **R//**
> Podemos ver que el overfitting comienza entre las epocas 10 y 15, que es cuando comienza a aumentar la perdida de validacion de 0.38 a 0.42. Pero definitivamente es antes de la 20 pues en esta la reduccion de la perdida de entrenamiento ya es minima (0.10) y ya se elevo considerablemente la perdida de validacion.

5. Proponga dos estrategias de regularización concretas (por ejemplo, Dropout o L2) que podría haber implementado para prevenir este comportamiento. Para cada una, explique intuitivamente qué fenómeno matemático está mitigando y qué impacto esperaría ver en las curvas de la tabla.
> **R//**
> Dropout. Esta tecnica nos ayuda a reducir la linealidad de un modelo, es decir, que sus predicciones se conviertan en simplemente regresiones lineales (lo cual es una posibilidad debido a la naturaleza de los calculos de un MLP). Podemos ver tambien en la perdida de entrenamiento que el patron es casi 1/2 por la perdida anterior, lo que podria indicar queel modelo encontro alguna relacion lineal para los datos especificamente de entrenamiento que no estan aplicando a los casos de validacion.

> L1. Esta tecnica se puede utilizar para llevar gradientes pequeñas directamente a 0, lo que genera esparcidad, es decir, menos neuronas tienden a activarse para un resultado, lo que se ha visto que conlleva a mejores generalizaciones en modelos.

6. Desde una perspectiva médica, ¿por qué es especialmente peligroso desplegar en producción un modelo que exhibe este patrón de sobreajuste para el diagnóstico de radiografías? Argumente más allá de los números.

> **R//**
> Un sistema overfitted poria llevar a muchas complicaciones. Por ejemplo, si por alguna razon el modelo se entreno en mas casos positivos de una enfermedad que requiere cirujia, si lo llevan a produccion, tendra muchos falsos positivos y llevara a cirujia a gente que no tenia complicaciones de ese timpo, afectandoles economicamente, en salud y emocionalmente.
> Siempre en redes neuronales existe el problema de la etica, especialmente, en como se relaciona con asignar responsabilidad sobre una decision. Hacer que un modelo de IA tome decisiones importantes sobre un paciente que podrian llevarlo a la mesa de cirugía y posiblemente hasta arriesgar su vida y salud, es un gran problema, pues es dificil justificar por qué la IA tomo esta decision. No lleva un hilo concreto de pensamiento basado en diagnosticos, sino es una caja negra que nos dice "si" o "no" sin decir como llego a esa conclusion.
> Aunque mucha gente siempre hace un esfuerzo por recalcar que las IAs nunca llegan a un 100% de accuracy, no se toma en cuenta que el problema no es que "no siempre estan en lo correcto" porque los humanos tambien cometen errores, sino que no hay bases concretas, justificables y comprensibles de sus resultados

### Pregunta 1.3
Un inversionista que asiste a la demo del proyecto le pregunta: "¿Por qué su modelo tiene un 94% de accuracy en las pruebas? ¿Eso significa que es confiable para diagnosticar neumonía?"

Usted sabe que el dataset de prueba contiene 700 radiografías normales y solo 150 radiografías con neumonía.

Responda lo siguiente en su reporte:

7. Calcule cuál sería el accuracy de un modelo naive que simplemente predice siempre 'Normal' para todas las imágenes. ¿Qué revela ese cálculo sobre el 94% reportado?
>*R//*
> 150/(150+700) = 17.6%. Eso significa que si el modelo siempre predijera NORMAL podria llegar a un accuracy del 82.3%.
> Esto revela que el modelo rinde mejor que un naive y es capaz de detectar al menos algunos casos de neumonía positiva. El nivel al que lo logre identificarlos correctamente dependeria ya de las metricas de f1, accuracy y recall especificamente de estos casos, pero por lo que es observable el modelo es decente


8. Explique por qué, en problemas médicos con clases desbalanceadas, métricas como el F1-Score o la Sensibilidad (Recall) para la clase minoritaria son más informativas que el accuracy. No se limite a definirlas; argumente su relevancia clínica.
>*R//*
> En problemasmedicos con clases desbalanceadas emtricas como el accuracy solo nos hablan sobre casos identificados sobre el total. Es decir, si solo hay 10 casos positivos de 100 y todos los tuviera mal el modelo, devolveria un accuracy del 90% lo que realmente no es representativo de la capacidad del modelo.
> Metricas como f1 y recall ya nos hablan sobre la proporcionalidad de las clases y sobre como son identificados estos. En lugr de decir "de todos los casos, cuales fueron exitosos" se pregunta de manera mas especifica "de los casos de neumonia, cuantos fueron verdaderos y falsos positivos" y a partir de eso da una metrica mas representativa de el exito de unmodelo frente a estas clases desbalanceadas. 

9. Como director técnico, ¿cómo le respondería al inversionista de forma honesta y profesional? Redacte una respuesta breve (3 a 5 oraciones) que sea técnicamente sólida pero comprensible para un no-especialista.

>*R//*
> El modelo tiene 94% de accuracy en las pruebas porque esta es una metrica que une los casos de neumonia positiva y negativa. Es decir, el modelo tiene alto porcentaje simplemente porque logro detectar que la mayoria de casos no son de neumonía. Esta metrica puede ser engañosa si lo que queremos es enfocarnos en las personas que sí tienen neumonia, de las cuales logra identificar correctmente el 70% de los casos.

> (el 70% se saco de el porcentaje de casos de neumonia, y los casos exitosos del modelo. 17.6% son casos de neumonia y los identifcados por el modelo son 12% [obtenido de el 82% del naive y el 94%] y si los dividimos 12/17 nos da cerca del 70%)

-----------------

## Task 2
Las siguientes preguntas evalúan su comprensión conceptual y estratégica del Transfer Learning y el Fine- Tuning, siempre enmarcadas en el contexto de MediScan Guatemala. Se valorará la solidez del argumento, la coherencia con la realidad operacional y la capacidad de ir más allá de la definición textual:


### Pregunta 2.1
Un desarrollador del equipo propone: "Para el modelo de radiografías no tiene sentido usar Transfer Learning desde ImageNet; ese dataset tiene fotos de perros, gatos y autos. Las radiografías son imágenes en escala de grises totalmente distintas. Mejor entrenamos desde cero para no contaminar el modelo.” Con esto responda en su reporte:

1. ¿Está de acuerdo con el desarrollador? Argumente su posición con base en lo que las capas convolucionales tempranas de una CNN realmente aprenden y por qué ese conocimiento sí es transferible, incluso entre dominios visualmente distintos.

2. Más allá del argumento técnico, ¿cuál es el costo real de entrenar desde cero para una startup como MediScan Guatemala? Considere al menos dos dimensiones distintas a la puramente computacional (p. ej., tiempo al mercado, riesgo, talento humano).

3. ¿Existe algún escenario legítimo en el que usted sí consideraría entrenar desde cero en lugar de hacer Transfer Learning? Descríbalo y justifíquelo. 

### Pregunta 2.2
Su equipo debe decidir la estrategia de Fine-Tuning para el modelo de MediScan Guatemala. El CTO les propone dos opciones

| Opción A | Congelar toda la red base B | Descongelar toda la red |
| --- | --- | --- |
| Capas base | Congeladas (gradiente = 0) | Todas entrenables |
| Tasa de aprendizaje | Alta (1e-3) | Muy baja (1e-5) en toda la red |
| Cabezal final | Entrenado desde cero  | Entrenado con tasa 1e-3 |

Responda en su reporte:

4. Dada la naturaleza del dataset médico disponible (800 imágenes, dominio distinto a ImageNet), ¿cuál opción recomendaría y por qué? No se limite a decir cuál es la regla de oro; explique la lógica detrás de esa regla en este caso concreto.

5. ¿Qué riesgo específico correría si aplica la Opción B con una tasa de aprendizaje alta (p. ej., 1e-3) en toda la red? ¿Cómo se llama ese fenómeno y cómo se evidenciaría durante el entrenamiento?

6. Si con el tiempo MediScan Guatemala logra recolectar 50,000 radiografías etiquetadas, ¿cambiaría su recomendación? Explique cómo evolucionaría su estrategia de Fine-Tuning y por qué.

### Pregunta 2.3
El área de Producto de MediScan Guatemala les presenta el siguiente requerimiento: "El hospital nos pide expandir el sistema para que también detecte fracturas en radiografías de muñeca, usando el mismo modelo que ya tenemos para neumonía. ¿Podemos reutilizarlo directamente o hay que empezar todo de nuevo?"

Responda en su reporte:

7. Evalúe la viabilidad de reutilizar el modelo de neumonía para el nuevo problema de fracturas en muñeca. ¿Qué partes del modelo podrían aprovecharse y cuáles habría que modificar? Argumente con base en lo que cada sección de la red ha aprendido.

8. Proponga un plan de acción concreto de tres pasos para adaptar el modelo existente al nuevo dominio. Sea específico: ¿qué congela, qué descongela, qué modifica en el cabezal, qué tasa de aprendizaje usaría y por qué?

9. ¿Qué riesgo ético o clínico implica reutilizar un modelo entrenado en un dominio para otro dominio médico sin una validación rigurosa? ¿Qué proceso de validación mínimo recomendaría antes de desplegarlo?


---------------------------
## Task 3
Utilice PyTorch o TensorFlow/Keras a su elección. No se proporciona código base; usted debe construir su solución apoyándose en la documentación oficial, recursos académicos y su criterio de ingeniería. Ejecute sus experimentos en Google Colab, Kaggle Notebooks o GPU local. Entregue el enlace al notebook con las celdas ejecutadas y los resultados visibles. El notebook debe estar limpio, comentado y reproducible. Las instrucciones a continuación describen las tareas que debe completar. La evaluación considera no solo que el código funcione, sino que usted entienda cada decisión que tomó y la justifique en su reporte. Con esto realice lo siguiente:

1. Preparación del Dataset (incluido en la evaluación global).
  - a. Descargue el dataset Chest X-Ray Images (Pneumonia) de Kaggle y cárguelo en su entorno de trabajo.
  - b. Divídalo en tres conjuntos: Entrenamiento (70%), Validación (15%) y Prueba (15%). Justifique en su reporte cómo realizó esta división y si respetó la distribución de clases (estratificación).
  - c. Implemente un pipeline de Data Augmentation apropiado para imágenes médicas. En su reporte, indique cada transformación elegida y argumente por qué es válida en este dominio. Asegúrese de que las transformaciones que aplique durante el entrenamiento sean diferentes a las del conjunto de validación y prueba; explique por qué esto importa.

2. Cargue dos modelos pre-entrenados en ImageNet de su elección (por ejemplo: ResNet50, VGG16, DenseNet121, MobileNetV2, EfficientNet, entre otros). Para cada uno:
  - a. Congele las capas base del modelo y reemplace el cabezal de clasificación para adaptarlo al problema binario (Normal vs. Neumonía). Justifique en su reporte por qué decidió congelar esas capas específicas.
  - b. Entrene ambos modelos usando la misma función de pérdida y optimizador. Documente los hiperparámetros que eligió (tasa de aprendizaje, épocas, tamaño de batch) y argumente por qué son razonables para este problema.
  - c. Implemente Early Stopping. Explique en su reporte qué métrica está monitoreando y cuál es la lógica detrás de detener el entrenamiento anticipadamente

3. Para cada modelo, calcule y registre las siguientes métricas en el conjunto de prueba
  - a. Accuracy (Exactitud)
  - b. F1-Score para la clase 'Neumonía' (clase positiva)
  - c. Sensibilidad / Recall para la clase 'Neumonía'
  - d. Tamaño del modelo guardado en disco (en MB)
  - e. Tiempo de inferencia promedio sobre 100 imágenes (en milisegundos)

4. Finalmente, redacte en su reporte un dictamen ejecutivo de 1 a 2 páginas que incluya:
  - a. Una tabla comparativa cruzando los dos modelos con todas las métricas registradas en el Paso 3.
  - b. Análisis Sensibilidad / Accuracy, cuál modelo detecta mejor neumonía y por qué se prefiere alta Sensibilidad aunque sacrifique Accuracy
  - c. Recomendación final al CTO, modelo para producción on-premise sin GPU
  - d. Reflexión sobre limitaciones del experimento
  - e. ¿Cuánto "cuesta" en MB cada punto porcentual de F1? ¿Vale la pena el modelo más pesado en el contexto de clínicas rurales con hardware limitado?
  - f. No solo reportar los ms, sino argumentar si ese tiempo es aceptable en un flujo clínico real (ej. ¿cuántas radiografías por hora podría procesar el sistema?).
  - g. Reflexión sobre si congelar vs. descongelar capas impactó los resultados obtenidos, conectando la práctica con lo analizado en el Task 2.
  - h. ¿Funcionaría el modelo igual con radiografías de equipos distintos, distintas poblaciones o distintos hospitales? ¿Qué implicaciones tiene eso para MediScan Guatemala?

----------------------
## Task 4
Como parte del laboratorio de esta semana, y en búsqueda que vayan complementado sus conocimientos
fuera de lo técnico, cada grupo deberá realizar lo siguiente:
Un integrante del grupo debe dejar un comentario en la publicación de LinkedIn del curso. El comentario
debe tener mínimo dos oraciones y debe conectar algo específico del laboratorio de esta semana con el
tema de la publicación. No hay una respuesta correcta. Se busca su reflexión genuina sobre cómo el
concepto técnico que trabajaron se relaciona con decisiones reales en organizaciones.
Los otros dos integrantes del grupo deben reaccionar a la publicación con cualquier reacción de su
preferencia.
El integrante que comenta debe tomar una captura de pantalla del comentario publicado y subirla como
parte de la entrega del laboratorio.
El link a la publicación será compartido por el profesor a través Discord.


https://www.linkedin.com/posts/albertosuriano_machinelearning-datascience-datastrategy-share-7439160493132587009-RtPu?utm_source=share&utm_medium=member_android&rcm=ACoAABLGaqcBzAwC5ioSJ7pQ_N512FZ6a_D3qDw

Requisitos del comentario:

LinkedIn es una red profesional real. Su comentario será visto por profesionales de datos, reclutadores y líderes de industria, no solo por su profesor. Esta es una oportunidad genuina de construir tu presencia profesional desde ahora, antes de graduarte.

Por eso el comentario debe leerse como la opinión de un profesional, no como la tarea de un estudiante. Mínimo dos oraciones. Sin mencionar la clase, el laboratorio ni que esto es una actividad académica.

Nadie contrata a alguien porque hizo una tarea. Pero sí recuerdan a quien dejó un comentario inteligente en el momento correcto.

Escribe algo que te gustaría que un reclutador o un líder de datos leyera sobre ti. Comparte una opinión, una observación o una pregunta real sobre el tema. Puede estar inspirado en lo que trabajaste esta semana, pero exprésalo como tu perspectiva profesional.

Ejemplo de un buen comentario:
"Este punto sobre los incentivos es exactamente lo que se ve en equipos que trabajan en silos. La parte técnica suele estar resuelta, pero nadie ha rediseñado cómo se comparte la información entre áreas."

Ejemplo de un mal comentario:
"En el laboratorio de esta semana trabajamos con esto y me pareció interesante."

El primer tipo de comentario construye tu reputación.

El segundo no es muy constructivo